# [LAB] Harness 실습: SKILL.md로 Agent 동작 바꾸기

Agent가 읽을 스킬 문서(SKILL.md)를 쓰고, 그 문서를 읽고 따르는 Agent를 만들어 실행합니다. 그다음 코드와 모델은 그대로 두고 SKILL.md 한 줄만 바꿔 Agent의 동작이 달라지는 것을 봅니다.

- **2장 Agent가 읽을 스킬 쓰기**: 삼성SDS 보고서 양식이 적힌 SKILL.md를 파일로 씁니다. AI에게 문서의 윗부분과 아랫부분이 각각 무슨 일을 하는지 설명받습니다.
- **3장 스킬을 읽는 Agent 만들기**: 모델·도구·작업 공간·스킬 폴더·시스템 프롬프트를 묶어 Agent를 만드는 보기를 고르고, 보고서 요청과 범위 밖 요청을 차례로 실행합니다.
- **4장 SKILL.md 고쳐 동작 바꾸기**: `# ✍️ CHANGE_HERE:` 줄의 스킬 설명을 바꾸고 Agent를 다시 실행해 결과를 비교합니다. AI와 함께 SKILL.md의 빠진 항목도 찾아봅니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. Agent를 실행하는 셀은 모델을 여러 번 호출하므로 결과가 나오기까지 수십 초가 걸리고, 답은 실행할 때마다 조금씩 다릅니다.

**목차**

- 1. 실습 방법
- 2. Agent가 읽을 스킬 쓰기
  - 2.1 SKILL.md 쓰기
- 3. 스킬을 읽는 Agent 만들기
  - 3.1 검색 도구
  - 3.2 Agent 만들기
  - 3.3 두 요청 실행하기
- 4. SKILL.md 고쳐 동작 바꾸기
  - 4.1 스킬 설명 바꿔 다시 실행하기
- 5. 과제: 스킬로 움직이는 Deep Agent
- 6. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. Agent가 읽을 스킬 쓰기

SKILL.md는 Agent가 특정 작업을 할 때 참고하는 절차 문서입니다. 이 실습의 스킬은 "삼성SDS 보고서 양식으로 보고서 쓰기"입니다. 문서는 두 부분입니다.

- **맨 위 `---` 사이**: 스킬 이름(`name`)과 설명(`description`). 설명에는 이 스킬을 **언제** 쓰는지 적습니다.
- **그 아래 본문**: 보고서 양식과 작성 규칙. 이 스킬을 **어떻게** 쓰는지 적습니다.

아래 셀을 실행하면 이 폴더 아래 `harness/skills/samsung-sds-report/SKILL.md` 파일이 만들어집니다. 다음 장에서 만들 Agent가 이 파일을 읽습니다. 모델은 호출하지 않습니다.

In [ ]:
from pathlib import Path

Path("harness/skills/samsung-sds-report").mkdir(parents=True, exist_ok=True)

### 2.1 SKILL.md 쓰기

In [ ]:
%%writefile harness/skills/samsung-sds-report/SKILL.md
---
name: samsung-sds-report
description: 삼성SDS AI연구소 공식 보고서 작성 스킬. 사용자가 "보고서", "리포트", "SDS 양식" 등을 언급하며 작성을 요청할 때 반드시 발동한다. 삼성SDS 내부 보고서 양식(SDS-AI-RPT 포맷)은 외부에 공개되지 않으므로 이 스킬 파일을 읽어야만 올바른 형식으로 작성할 수 있다.
---

# samsung-sds-report

## 삼성SDS AI연구소 보고서 작성 지침

이 스킬이 발동되면 아래 SDS-AI-RPT 양식을 반드시 사용합니다.
이 양식은 삼성SDS 내부 표준으로, 다른 형식으로 작성하면 안 됩니다.

---

## SDS-AI-RPT 보고서 양식

보고서는 아래 템플릿을 그대로 따릅니다. 섹션 코드([SDS-...])는 생략하지 않습니다.

```
[SDS-AI-RPT v2.1]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
보안등급: INTERNAL  |  배포범위: AI연구소 내부
작성일: YYYY-MM-DD  |  문서번호: AI-RPT-YYYYMM-NNN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

■ TL;DR (3줄 요약)
① [핵심 발견 1]
② [핵심 발견 2]
③ [권고 사항]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[SDS-1.0] 개요
  1.1 작성 배경
  [배경 설명]
  
  1.2 조사 범위
  [범위 설명]

[SDS-2.0] 현황 분석
  2.1 핵심 동향
  | 구분 | 내용 | SDS 영향도 |
  |------|------|-----------|
  | [항목] | [내용] | 상/중/하 |

  2.2 주요 플레이어
  [내용]

[SDS-3.0] 결론 및 제언
  3.1 결론
  [결론]
  
  3.2 SDS 대응 방향 (권고)
  - [권고 1]
  - [권고 2]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[SDS-END] 개정 이력
| 버전 | 일자 | 변경 내용 |
|------|------|---------|
| v1.0 | YYYY-MM-DD | 최초 작성 |
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

## 작성 규칙

1. `[SDS-AI-RPT v2.1]` 헤더로 시작합니다.
2. `■ TL;DR` 섹션은 반드시 3줄로 작성합니다.
3. 섹션 코드 `[SDS-1.0]`, `[SDS-2.0]`, `[SDS-3.0]` 은 생략하지 않습니다.
4. `[SDS-2.0]` 현황 분석에는 반드시 "SDS 영향도(상/중/하)" 컬럼이 있는 표를 포함합니다.
5. `[SDS-END]` 개정 이력 표로 마무리합니다.

**성공 기준**

- 🟢 셀 아래에 파일을 썼다는 한 줄(Writing … 또는 Overwriting …)이 출력되었다

위 셀을 실행하면 Agent가 읽을 SKILL.md 파일이 만들어집니다. 이 장에서는 파일만 씁니다. 다음 장에서 Agent를 만들어 실행합니다.

**볼 지점**
- 설명(`description`)에 "보고서", "리포트", "SDS 양식"을 언급하며 작성을 요청할 때 쓰라고 적혀 있습니다. Agent는 처음에 이 이름과 설명만 보고, 필요하다고 판단하면 파일 전체를 읽습니다.
- 본문의 양식은 `[SDS-AI-RPT v2.1]` 헤더로 시작합니다. 다음 장에서 Agent의 답이 이 헤더로 시작하는지 보면, Agent가 스킬을 따랐는지 알 수 있습니다.
- 작성 규칙 다섯 줄 중 다음 장 출력에서 확인할 수 있는 것은 첫 줄(헤더)입니다. 출력은 앞 200자까지만 보이기 때문입니다.

## 3. 스킬을 읽는 Agent 만들기

`create_deep_agent`로 Agent를 만듭니다. 이 함수는 Agent가 일하는 환경을 한 번에 묶습니다.

- **모델**과 **도구**: 검색 도구 `search_web`. 실제로 검색하지 않고 정해 둔 글(검색 결과 흉내)을 돌려줍니다.
- **작업 공간** `FilesystemBackend`: Agent가 파일을 읽고 쓸 수 있는 폴더입니다. 이 실습에서는 `harness` 폴더만 Agent에게 보여 줍니다. Agent에게는 이 폴더가 맨 위(`/`)로 보여서, SKILL.md는 `/skills/samsung-sds-report/SKILL.md`로 보입니다.
- **스킬 폴더** `skills`: 스킬 폴더들을 담은 상위 폴더(`skills/`)입니다. Agent는 그 아래 폴더마다 든 SKILL.md를 찾습니다.
- **시스템 프롬프트**: 범위를 벗어난 요청을 거절하게 합니다. 스킬은 보고서 요청을 어떻게 처리할지만 알려 주고, 보고서와 관계없는 요청을 막지는 않기 때문입니다.
- **체크포인터** `MemorySaver`: 대화 상태를 기억합니다.

마지막 셀은 두 요청을 차례로 보냅니다. 사례 1은 삼성SDS 양식 보고서 요청, 사례 2는 범위를 벗어난 퀵소트 코드 요청입니다. 출력은 Agent가 한 단계를 마칠 때마다 한 줄씩 나옵니다. 줄 앞의 `[HumanMessage]`는 사람의 요청, `[AIMessage]`는 모델의 답, `[ToolMessage]`는 도구 실행 결과입니다.

**왜 배우나:** 하네스는 모델에게 더 긴 지시문을 주는 것이 아니라, 모델이 일할 환경을 만드는 일입니다. 이 셀 하나에 모델·도구·작업 공간·규칙·기억이 모두 들어갑니다.

준비 코드입니다. 필요한 도구를 불러오고, Agent의 작업 폴더를 `harness`로 정합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
Harness Engineering 실습

deepagents의 create_deep_agent + skills 디렉토리로 Agent를 빌드하는 예제.

skills/samsung-sds-report/SKILL.md 가 Agent의 행동 규칙을 정의한다.
Agent는 description을 보고 이 스킬이 필요한지 판단한 뒤 SKILL.md를 읽어 지시사항을 따른다.

→ SKILL.md만 수정하면 모델·코드 변경 없이 Agent 동작을 조정할 수 있다.
"""

from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from config.llm import get_llm

load_dotenv()

HARNESS_DIR = Path("harness")  # Agent의 작업 폴더. SKILL.md가 이 폴더 아래에 있습니다

### 3.1 검색 도구

In [ ]:
# ── 도구 정의 ─────────────────────────────────────────────────────────────────


@tool
def search_web(query: str) -> str:
    """웹에서 주제를 검색한다."""
    # 실제 구현에서는 Tavily, SerpAPI 등을 사용한다.
    return (
        f"[검색: {query}]\n"
        f"- 최근 동향: 빠른 성장세, 기업 도입 증가\n"
        f"- 핵심 기술: 트랜스포머 아키텍처, RLHF, RAG\n"
        f"- 주요 사례: 고객 지원 자동화, 코드 리뷰, 문서 요약\n"
        f"- 시장 규모: 2024년 기준 약 2,000억 달러 (연 30% 성장 예상)"
    )

### 3.2 Agent 만들기

**✅ LLM, 사용 가능한 도구, 백엔드, 스킬, 시스템 프롬프트를 이용해 Deep Agent를 정의해 주세요.**

- 🟢 "두 요청 실행하기" 셀에서 사례 1의 마지막 `[AIMessage]`가 `[SDS-AI-RPT v2.1]`로 시작하고, 사례 2(퀵소트)는 코드 없이 거절한다
- ⛔ 사례 1이 SDS 양식이 아니거나 사례 2에서 퀵소트 코드가 나온다 → 다른 보기를 골라 이 셀과 "두 요청 실행하기" 셀을 다시 실행합니다

In [ ]:
# ── Agent 생성 ────────────────────────────────────────────────────────────────
# FilesystemBackend: 로컬 파일시스템을 Agent의 작업 공간으로 사용
# skills: 스킬 폴더들을 담은 상위 디렉토리 목록 (backend root_dir 기준 상대경로). 그 아래 폴더마다 SKILL.md가 있다
#   → Agent는 SKILL.md의 name+description을 보고 "이 스킬이 필요하다" 판단 후 read_file로 상세 지시 로드

# virtual_mode=True: 에이전트에게 가상 경로(/skills/... 등)로 파일시스템을 노출
# ls 결과가 Windows 절대경로 대신 /skills/ 같은 가상경로로 반환되어 경로 혼동 없음
backend = FilesystemBackend(root_dir=str(HARNESS_DIR), virtual_mode=True)

# ✅ CHOOSE_HERE: LLM, 사용 가능한 도구, 백엔드, 스킬, 시스템 프롬프트를 이용해 Deep Agent를 정의해 주세요.
# 이때, 시스템 프롬프트를 이용해 범위를 벗어나지 않도록 해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# agent = create_deep_agent(                                  # 1️⃣ skills에 스킬 하나의 폴더를 넣음
#     model=get_llm(),                                        # 1️⃣
#     tools=[search_web],                                     # 1️⃣
#     backend=backend,                                        # 1️⃣
#     skills=["skills/samsung-sds-report/"],                  # 1️⃣
#     system_prompt="코드 작성, 수학 계산 등 보고서·리서치 범위를 벗어난 요청은 정중히 거절한다.",  # 1️⃣
#     checkpointer=MemorySaver(),                             # 1️⃣
# )                                                           # 1️⃣
#
# agent = create_deep_agent(                                  # 2️⃣ system_prompt 없이 만듦
#     model=get_llm(),                                        # 2️⃣
#     tools=[search_web],                                     # 2️⃣
#     backend=backend,                                        # 2️⃣
#     skills=["skills/"],                                     # 2️⃣
#     checkpointer=MemorySaver(),                             # 2️⃣
# )                                                           # 2️⃣
#
# agent = create_deep_agent(                                  # 3️⃣ skills에 상위 폴더 skills/를 넣음
#     model=get_llm(),                                        # 3️⃣
#     tools=[search_web],                                     # 3️⃣
#     backend=backend,                                        # 3️⃣
#     skills=["skills/"],                                     # 3️⃣
#     system_prompt="코드 작성, 수학 계산 등 보고서·리서치 범위를 벗어난 요청은 정중히 거절한다.",  # 3️⃣
#     checkpointer=MemorySaver(),                             # 3️⃣
# )                                                           # 3️⃣
#
# agent = create_deep_agent(                                  # 4️⃣ backend 없이 만듦
#     model=get_llm(),                                        # 4️⃣
#     tools=[search_web],                                     # 4️⃣
#     skills=["skills/"],                                     # 4️⃣
#     system_prompt="코드 작성, 수학 계산 등 보고서·리서치 범위를 벗어난 요청은 정중히 거절한다.",  # 4️⃣
#     checkpointer=MemorySaver(),                             # 4️⃣
# )                                                           # 4️⃣

<details><summary>정답 보기</summary>

네 보기는 모델(`get_llm()`)·도구(`search_web`)·체크포인터(`MemorySaver()`)가 같고, 작업 공간(`backend`)·스킬 폴더(`skills`)·시스템 프롬프트(`system_prompt`) 줄이 조금씩 다릅니다. 이 셀은 Agent만 만들고 아무것도 출력하지 않습니다. 네 보기 모두 오류 없이 실행되어, 맞게 골랐는지는 "두 요청 실행하기" 셀의 출력에서만 드러납니다.

**정답: 3️⃣**

- 1️⃣ `skills`에는 스킬 폴더들을 담은 상위 폴더를 넣습니다. SKILL.md가 든 폴더를 직접 넣으면 그 아래에서 스킬 폴더를 찾다가 하나도 찾지 못합니다. 오류 없이 실행되지만 사례 1에 SKILL.md를 읽는 줄이 없고, 마지막 답이 `[SDS-AI-RPT v2.1]`로 시작하지 않습니다(일반 보고서 형식으로 씁니다).
- 2️⃣ 스킬은 보고서 요청을 어떻게 처리할지만 알려 줍니다. 범위 밖 요청을 막는 규칙이 없어, 사례 1은 SDS 양식으로 잘 쓰지만 사례 2에서 퀵소트 코드를 그대로 써 줍니다.
- 3️⃣ `harness` 폴더를 작업 공간으로 주고, 그 안의 `skills/` 아래에서 SKILL.md를 찾게 합니다. 사례 1은 SKILL.md를 읽고 SDS 양식으로 쓰고, 사례 2는 시스템 프롬프트에 따라 도구 없이 거절합니다.
- 4️⃣ `backend`를 빼면 Agent는 `harness` 폴더가 아니라 비어 있는 기본 작업 공간을 봅니다. `skills/` 폴더가 거기에는 없어 스킬을 하나도 찾지 못합니다. 오류 없이 실행되지만 사례 1에 SKILL.md를 읽는 줄이 없고, 마지막 답이 `[SDS-AI-RPT v2.1]`로 시작하지 않습니다(SDS 내부 양식은 재현할 수 없다고 적기도 합니다).

</details>

### 3.3 두 요청 실행하기

In [ ]:
# ── 대화형 실행 ───────────────────────────────────────────────────────────────


def chat(session_id: str, message: str) -> str:
    # recursion_limit: LangGraph 노드 실행 횟수 상한 — 기본값 25, 무한루프 방지
    config = {"configurable": {"thread_id": session_id}, "recursion_limit": 40}
    response = ""
    for chunk in agent.stream(
        {"messages": [HumanMessage(content=message)]},
        config=config,
        stream_mode="values",
    ):
        last_msg = chunk["messages"][-1]
        msg_type = type(last_msg).__name__
        content = last_msg.content or ""
        # 각 단계마다 메시지 타입과 내용 앞부분을 출력해 어디서 멈추는지 확인
        print(f"[{msg_type}] {content[:200]}", flush=True)
    print()
    return response


print("=== Harness Agent (skills/ 디렉토리 기반) ===")
print(f"스킬 경로: {HARNESS_DIR / 'skills'}\n")

# 케이스 1: SDS 보고서 양식을 명시 → 모델이 SKILL.md를 읽어야만 형식을 알 수 있음
# 케이스 2: 범위 이탈 → system_prompt로 거절
test_cases = [
    ("session_1", "AI Agent 최신 트렌드를 삼성SDS 보고서 양식으로 작성해줘."),
    ("session_2", "파이썬으로 퀵소트 코드를 짜줘."),
]

for session_id, message in test_cases:
    print(f"사용자: {message}")
    print("Agent: ", end="", flush=True)
    chat(session_id, message)
    print()

**볼 지점**
- 사례 1의 흐름: `[AIMessage]` 줄이 비어 있으면 모델이 글 대신 도구 호출을 보낸 단계입니다. 그 아래 `[ToolMessage]`가 도구 실행 결과입니다. 먼저 SKILL.md를 읽고(`@@ lines 1-…` 아래에 파일 내용), 검색을 몇 번 한 뒤, 마지막 `[AIMessage]`에서 보고서를 씁니다. 각 줄은 앞 200자까지만 출력합니다.
- 사례마다 `[HumanMessage]` 줄이 두 번 나옵니다. 처음 받은 요청을 한 번, 스킬 목록을 읽는 준비 단계를 마친 뒤 한 번 더 출력한 것입니다. 요청이 두 번 간 것은 아닙니다.
- 사례 2는 도구를 호출하지 않고 바로 거절합니다. 스킬이 아니라 시스템 프롬프트가 막은 것입니다.
- 검색 결과는 정해 둔 글이라 늘 같습니다. 검색어와 검색 횟수는 실행할 때마다 달라집니다.

## 4. SKILL.md 고쳐 동작 바꾸기

코드와 모델은 그대로 두고 SKILL.md만 바꿔 Agent의 동작이 달라지는지 봅니다. 아래 셀은 2장의 SKILL.md와 같고, `# ✍️ CHANGE_HERE:` 표시만 더했습니다. 맨 위 `---` 사이에서는 `#`으로 시작하는 줄이 주석이라 Agent가 읽는 내용에 영향이 없습니다.

한 번 바꿔 볼 때마다 세 셀을 차례로 실행합니다.

1. 아래 셀을 실행해 SKILL.md를 다시 씁니다.
2. 3장의 **Agent 만들기** 셀을 다시 실행합니다. Agent는 대화를 시작할 때 스킬 설명을 한 번 읽어 기억해 두므로, Agent를 새로 만들어야 바뀐 설명을 읽습니다.
3. 3장의 **두 요청 실행하기** 셀을 다시 실행하고, 바꾸기 전 결과와 비교합니다.

**왜 배우나:** 스킬 설명은 Agent가 이 스킬을 쓸지 판단하는 근거입니다. 코드를 고치지 않고 문서만 고쳐도 Agent의 동작이 바뀝니다.

In [ ]:
from pathlib import Path

Path("harness/skills/samsung-sds-report").mkdir(parents=True, exist_ok=True)

### 4.1 스킬 설명 바꿔 다시 실행하기

**✍️ CHANGE_HERE: 스킬을 언제 쓰는지 적은 설명을 바꿔 봅니다**

먼저 그대로 실행해 결과가 3장과 같은지 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 `description:` 뒤의 글을 바꿉니다. 예: `회의록 요약 스킬. 보고서 작성 요청에는 쓰지 않는다.` 설명 안에는 콜론(`:`)을 쓰지 않습니다. 요청에 이미 "SDS 양식"이 있어 다른 문구로는 결과가 바뀌지 않을 수 있습니다. 처음에는 이 예시 그대로 바꿔 봅니다.

In [ ]:
%%writefile harness/skills/samsung-sds-report/SKILL.md
---
name: samsung-sds-report
# ✍️ CHANGE_HERE: 스킬을 언제 쓰는지 적은 설명을 바꿔 봅니다
description: 삼성SDS AI연구소 공식 보고서 작성 스킬. 사용자가 "보고서", "리포트", "SDS 양식" 등을 언급하며 작성을 요청할 때 반드시 발동한다. 삼성SDS 내부 보고서 양식(SDS-AI-RPT 포맷)은 외부에 공개되지 않으므로 이 스킬 파일을 읽어야만 올바른 형식으로 작성할 수 있다.
---

# samsung-sds-report

## 삼성SDS AI연구소 보고서 작성 지침

이 스킬이 발동되면 아래 SDS-AI-RPT 양식을 반드시 사용합니다.
이 양식은 삼성SDS 내부 표준으로, 다른 형식으로 작성하면 안 됩니다.

---

## SDS-AI-RPT 보고서 양식

보고서는 아래 템플릿을 그대로 따릅니다. 섹션 코드([SDS-...])는 생략하지 않습니다.

```
[SDS-AI-RPT v2.1]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
보안등급: INTERNAL  |  배포범위: AI연구소 내부
작성일: YYYY-MM-DD  |  문서번호: AI-RPT-YYYYMM-NNN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

■ TL;DR (3줄 요약)
① [핵심 발견 1]
② [핵심 발견 2]
③ [권고 사항]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[SDS-1.0] 개요
  1.1 작성 배경
  [배경 설명]
  
  1.2 조사 범위
  [범위 설명]

[SDS-2.0] 현황 분석
  2.1 핵심 동향
  | 구분 | 내용 | SDS 영향도 |
  |------|------|-----------|
  | [항목] | [내용] | 상/중/하 |

  2.2 주요 플레이어
  [내용]

[SDS-3.0] 결론 및 제언
  3.1 결론
  [결론]
  
  3.2 SDS 대응 방향 (권고)
  - [권고 1]
  - [권고 2]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[SDS-END] 개정 이력
| 버전 | 일자 | 변경 내용 |
|------|------|---------|
| v1.0 | YYYY-MM-DD | 최초 작성 |
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

## 작성 규칙

1. `[SDS-AI-RPT v2.1]` 헤더로 시작합니다.
2. `■ TL;DR` 섹션은 반드시 3줄로 작성합니다.
3. 섹션 코드 `[SDS-1.0]`, `[SDS-2.0]`, `[SDS-3.0]` 은 생략하지 않습니다.
4. `[SDS-2.0]` 현황 분석에는 반드시 "SDS 영향도(상/중/하)" 컬럼이 있는 표를 포함합니다.
5. `[SDS-END]` 개정 이력 표로 마무리합니다.

**성공 기준**

- 🟢 셀 아래에 파일을 다시 썼다는 한 줄(Overwriting …)이 출력되었다
- 🟢 예시 문구로 바꾼 뒤 Agent 만들기 셀과 두 요청 실행하기 셀을 다시 실행하면, 사례 1의 마지막 `[AIMessage]`가 `[SDS-AI-RPT v2.1]`로 시작하지 않는다(다른 형식으로 쓰거나, 그 양식으로는 쓸 수 없다고 답한다)
- ⛔ 결과가 바꾸기 전과 똑같다 → 바꾼 설명이 위 예시 그대로인지, Agent 만들기 셀부터 다시 실행했는지 확인합니다. 그래도 같으면 Agent 만들기 셀부터 한 번 더 실행합니다. 모델이 판단하므로 매번 같지는 않습니다
- ⛔ 바꾼 설명 안에 콜론(`:`)이 있다 → 콜론이 들어가면 Agent가 스킬을 아예 찾지 못해, 설명 내용과 관계없이 양식이 사라집니다. 콜론을 빼고 위 셀부터 다시 실행합니다

SKILL.md를 다시 썼으면 3장으로 올라가 Agent 만들기 셀과 두 요청 실행하기 셀을 차례로 다시 실행합니다.

**볼 지점**
- 바꾼 것은 SKILL.md의 설명 한 줄뿐입니다. 모델도, Agent를 만드는 코드도 그대로입니다.
- 설명을 바꿔도 Agent가 SKILL.md를 읽는 단계는 남아 있을 수 있습니다. 요청에 "삼성SDS 보고서 양식"이 들어 있어, 이름이 비슷한 스킬을 열어 본 것으로 보입니다. 이 경우에도 예시 설명은 "보고서 작성 요청에는 쓰지 않는다"고 하므로, 읽은 뒤 양식을 따르지 않는 결과가 나옵니다.
- 원래대로 되돌리려면 2장의 SKILL.md 쓰기 셀을 다시 실행하고, Agent 만들기 셀부터 다시 실행합니다.

## 5. 과제: 스킬로 움직이는 Deep Agent

이 장의 학습 목표를 코드로 구현해 봅니다. 과제 파일과 자세한 안내(할 일, 프롬프트 힌트, 실행 방법)는 노트북 밖 폴더에 있습니다.

**학습 목표**

- `@tool` 도구와 시스템 프롬프트를 `create_deep_agent`에 묶어 Agent를 만들 수 있다
- 머리말(`name`·`description`)과 양식 본문이 있는 SKILL.md를 쓰고 skills 폴더로 연결해 그 양식대로 답하게 할 수 있다
- `system_prompt`로 범위 밖 요청을 거절시키고 두 요청의 결과를 비교할 수 있다
- 코드·모델은 그대로 두고 SKILL.md 한 줄만 바꿔 출력이 달라지는 것을 전후 비교로 보일 수 있다

**터미널 과제**

1. VS Code 탐색기에서 `course/Chapter02/tasks/4-1/README.md`를 열고 안내대로 AI와 함께 작성합니다.
2. VS Code 메뉴 **터미널 → 새 터미널**을 열고(저장소 맨 위 폴더에서 열립니다) 아래 명령으로 실행합니다.

```
uv run --env-file .env python course/Chapter02/tasks/4-1/4-1_task.py
```

3. 다 됐으면 AI에게 **"4-1 과제 채점해 줘"**라고 말합니다. 학습 목표마다 🟢 통과 / ⛔ 수정 필요 / ❔ 확인 불가와 다음 할 일이 나옵니다.

## 6. 정리

**이 노트북에서 한 것**
- Agent가 읽을 스킬 문서(SKILL.md)를 파일로 썼습니다. 맨 위 설명은 스킬을 **언제** 쓰는지, 본문은 **어떻게** 쓰는지 알려 줍니다.
- `create_deep_agent`로 모델·도구·작업 공간·스킬 폴더·시스템 프롬프트·체크포인터를 묶어 Agent를 만들었습니다.
- 보고서 요청에는 Agent가 SKILL.md를 읽고 SDS 양식으로 썼고, 범위 밖 요청은 시스템 프롬프트에 따라 거절했습니다.
- 코드와 모델은 그대로 두고 SKILL.md 설명 한 줄만 바꿔 Agent의 동작이 달라지는 것을 보았습니다.
- AI와 함께 SKILL.md를 좋은 스킬의 기본 구조와 비교해 빠진 항목을 찾고 보강했습니다.

보기를 골라 실행하고, 결과를 성공 기준(🟢/⛔)과 비교한 뒤 정답 보기의 해설로 확인했습니다.